# Jalon 2 — inv-2-MR-Sort : version finale

On veut apprendre **deux modèles MR-Sort**, un par segment de clients, sans savoir à quel segment appartient chaque objet. Il faut donc trouver en même temps la partition du learning set et les deux modèles. Chaque modèle est appris avec l'heuristique de `lincs`.

Ce notebook regroupe ce qui a été retenu des trois notebooks exploratoires (`jalon2_2mrsort`, `jalon2_ambigus`, `jalon2_inv2mrsort`) :
- section 3 : l'algorithme et la justification de chaque choix ;
- section 5 : le benchmark, avec un jeu de test ;
- section 6 : une cellule pour lancer l'algorithme sur un fichier ou sur un dossier.

**Hypothèses**
- Les features sont dans $[0, 1]$ et à maximiser.
- Les catégories sont ordonnées de la pire à la meilleure.
- Il y a 2 segments, de tailles inconnues.
- L'algorithme est **aveugle** : il ne connaît ni la proportion des segments, ni le niveau de bruit. Les champs `rep` et `noise` du nom de fichier ne servent qu'à l'évaluation.

In [2]:
import re, time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import lincs, lincs.classification as C

DATA_DIR = Path("Datasets-Step2")

ITER_SECONDS   = 1.0     # temps lincs par apprentissage, pendant les itérations (pour 6 critères / 3 catégories)
POLISH_SECONDS = 4.0     # temps lincs de l'apprentissage final (polish), même référence
MAX_ITER       = 20      # nombre maximal d'itérations par départ
INITS          = ("errors", "random", "random", "random")   # départs : 1 guidé + 3 aléatoires
MIN_FRAC       = 0.10    # taille minimale d'un segment pour apprendre son modèle
print("lincs", lincs.__version__)

lincs 2.0.2


## 1. Chargement des données

On lit `problem.yml` avec `yaml` (le fichier fourni n'a pas la clé `kind` qu'attend lincs 2) et le CSV avec pandas. La classe `y` d'un objet est l'indice de sa catégorie, de 0 (la pire) à $p-1$. `file_info` extrait du nom de fichier les informations qui servent **uniquement à l'évaluation**.

In [3]:
def load_dataset(csv_path):
    csv_path = Path(csv_path)
    pb = yaml.safe_load((csv_path.parent / "problem.yml").read_text())
    assert all(c.get("preference_direction", "increasing") == "increasing" for c in pb["criteria"])
    criteria = [str(c["name"]) for c in pb["criteria"]]
    categories = [str(c["name"]) for c in pb["ordered_categories"]]
    df = pd.read_csv(csv_path)
    y = df["category"].astype(str).map({c: h for h, c in enumerate(categories)})
    assert not y.isna().any(), "catégorie inconnue"
    return df[criteria].to_numpy(float), y.to_numpy(int), len(categories)

def file_info(csv_path):
    csv_path = Path(csv_path)
    info = dict(type=csv_path.parent.parent.name, dossier=csv_path.parent.name, fichier=csv_path.stem)
    m = re.search(r"n(\d+)_rep(\d+)-(\d+)_noise(\d+)", csv_path.name)
    if m is None:                                     # nom hors convention : pas d'infos d'évaluation
        return dict(info, N=np.nan, rep="?", petit_attendu=np.nan, bruit=np.nan)
    return dict(info, N=int(m[1]), rep=f"{m[2]}-{m[3]}", petit_attendu=min(int(m[2]), int(m[3])) / 100,
                bruit=int(m[4]) / 100)

## 2. Briques MR-Sort autour de lincs

- `learn_mrsort` appelle l'heuristique *weights-profiles-breed* de lincs : une population de 9 modèles, dont les 4 moins bons sont réinitialisés à chaque itération. L'apprentissage s'arrête à 100 % de précision, quand le temps alloué est écoulé, ou après 30 itérations sans progrès.
- **Temps alloué** (`budget`) : il est proportionnel au nombre de paramètres du modèle, $n(p-1) + n + 1$. *Pourquoi :* un modèle à 10 critères et 5 catégories a 2,7 fois plus de paramètres qu'un modèle à 6 critères et 3 catégories. Ce calcul n'utilise que $n$ et $p$, qui sont des entrées du problème.
- `predict` et `margins` appliquent la règle MR-Sort en numpy, en float32 comme lincs. *Pourquoi :* c'est bien plus rapide que de repasser par lincs, et on obtient exactement les mêmes affectations (vérifié ci-dessous). La marge sert à la réaffectation.

**Marge** d'un objet $x$ de catégorie $h$ : $\;m(x) = \min\big(C(x, b^{h-1}) - \lambda,\ \lambda - C(x, b^{h})\big)$, où $C(x, b)$ est la somme des poids des critères sur lesquels $x$ atteint le profil $b$. Elle est positive si $x$ est bien classé, et d'autant plus grande que ce classement est solide. On la calcule avec des poids normalisés (somme = 1) pour pouvoir comparer les deux modèles.

In [4]:
def n_params(n, p):
    return n * (p - 1) + n + 1

def budget(base_seconds, n, p):
    return base_seconds * n_params(n, p) / n_params(6, 3)

def make_problem(n, p):
    crits = [C.Criterion(f"c{i + 1}", C.Criterion.RealValues(C.Criterion.PreferenceDirection.increasing, 0., 1.))
             for i in range(n)]
    return C.Problem(crits, [C.Category(f"cat{h}") for h in range(p)])

def to_alts(problem, X, y=None):
    return C.Alternatives(problem, [
        C.Alternative(f"a{k}", [C.Performance(C.Performance.Real(float(v))) for v in X[k]],
                      None if y is None else int(y[k])) for k in range(len(X))])

def learn_mrsort(X, y, p, seed=0, seconds=2.0, models=9, patience=30):
    problem = make_problem(X.shape[1], p)
    pre = C.PreprocessedLearningSet(problem, to_alts(problem, X, y))
    mbl = C.LearnMrsortByWeightsProfilesBreed.ModelsBeingLearned(pre, models, int(seed))
    init = C.InitializeProfilesForProbabilisticMaximalDiscriminationPowerPerCriterion(pre, mbl)
    weights = C.OptimizeWeightsUsingGlop(pre, mbl)
    profiles = C.ImproveProfilesWithAccuracyHeuristicOnCpu(pre, mbl)
    breed = C.ReinitializeLeastAccurate(mbl, init, weights, count=models // 2)
    term = C.TerminateWhenAny([C.TerminateAtAccuracy(mbl, len(X)), C.TerminateAfterSeconds(seconds),
                               C.TerminateAfterIterationsWithoutProgress(mbl, patience)])
    m = C.LearnMrsortByWeightsProfilesBreed(pre, mbl, init, weights, profiles, breed, term).perform()
    B = np.array([[np.inf if av.real_thresholds.thresholds[h] is None else av.real_thresholds.thresholds[h]
                   for av in m.accepted_values] for h in range(p - 1)])          # (p-1, n)
    W = np.array(m.sufficient_coalitions[0].weights.criterion_weights)           # seuil de majorité = 1 (lincs)
    return {"profiles": B, "raw_weights": W, "weights": W / W.sum(), "lmbda": 1. / W.sum(),
            "_lincs": m, "_problem": problem}

def concordance(model, X):
    # C[k, h] = somme des poids (bruts) des critères où x_k >= b^h, en float32 comme lincs
    X32 = np.asarray(X, dtype=np.float32)
    W32 = np.asarray(model["raw_weights"], dtype=np.float32)
    return np.stack([(X32 >= b.astype(np.float32)).astype(np.float32) @ W32 for b in model["profiles"]],
                    axis=1).astype(float)

def predict(model, X):
    return np.cumprod(concordance(model, X) >= 1., axis=1).sum(axis=1)       # nb de frontières franchies

def margins(model, X, y):
    Cn = (concordance(model, X) - 1.) / model["raw_weights"].sum()          # C(x, b^h) - lambda, poids normalisés
    m = np.full(len(X), np.inf)
    low = y >= 1                                                             # doit franchir b^{h-1}
    m[low] = np.minimum(m[low], Cn[low, y[low] - 1] + 1e-12)
    up = np.where(y <= Cn.shape[1] - 1)[0]                                   # ne doit pas franchir b^h
    m[up] = np.minimum(m[up], -Cn[up, y[up]] - 1e-12)
    m[np.isinf(m)] = 1.
    return m

def show_model(model, title):
    df = pd.DataFrame(model["profiles"], index=[f"b{h + 1}" for h in range(len(model["profiles"]))],
                      columns=[f"c{i + 1}" for i in range(model["profiles"].shape[1])])
    df.loc["poids"] = model["weights"]
    print(f"--- {title}   lambda = {model['lmbda']:.3f}")
    display(df.round(3))

In [5]:
# Vérification : la prédiction numpy donne les mêmes catégories que lincs
X, y, p = load_dataset(DATA_DIR / "type_a/cr06_ca03/n0300_rep50-50_noise00.csv")
m = learn_mrsort(X, y, p, seed=1, seconds=1.0)
alts = to_alts(m["_problem"], X)
C.classify_alternatives(m["_problem"], m["_lincs"], alts)
print("accord numpy / lincs :", np.mean([a.category_index for a in alts.alternatives] == predict(m, X)))

accord numpy / lincs : 1.0


## 3. Algorithme inv-2-MR-Sort

C'est une alternance de type k-means, ou *clusterwise regression* (Späth 1979). On apprend un modèle par segment, puis on réaffecte chaque objet au modèle qui l'explique le mieux, et on recommence.

```
pour chaque départ (1 guidé + 3 aléatoires) :
    partition initiale
    répéter au plus MAX_ITER fois :
        apprendre un modèle lincs par segment
        réaffecter chaque objet au modèle qui l'explique strictement mieux (écart de catégorie, puis marge)
        s'arrêter si aucun objet n'a changé de segment
    garder la meilleure itération, puis réapprendre chaque modèle plus longtemps (polish)
retenir le départ qui explique le plus d'objets
```

| Choix | Ce qu'on fait | Pourquoi |
|---|---|---|
| **Départs** | 1 départ guidé (les objets mal classés par un modèle unique forment le segment 1) + 3 départs aléatoires | L'alternance converge vers un optimum local qui dépend du départ. Plusieurs départs sont le remède standard. Le départ guidé part d'une partition informative : un modèle unique s'ajuste surtout au comportement dominant, et ses erreurs concentrent les objets de l'autre comportement. |
| **Réaffectation** | ordre lexicographique : d'abord l'écart de catégorie $\lvert h_{prédit} - h \rvert$, puis la marge | Pour un objet mal classé par les deux modèles, se tromper d'une catégorie vaut mieux que de deux. À égalité, la marge départage selon la solidité du classement. |
| **Déplacement** | un objet ne change de segment que s'il est **strictement** mieux expliqué par l'autre modèle ; en cas d'égalité, il reste où il est | C'est la règle de Späth et de l'algorithme ACC : elle garantit la convergence et évite les allers-retours. Envoyer les égalités vers un segment fixe créerait un déséquilibre artificiel, car les égalités de marge sont fréquentes. |
| **Segment trop petit** | si un segment fait moins de 10 % des objets, son modèle est appris sur un tirage aléatoire de 10 % des objets | Évite d'apprendre un modèle sur une poignée d'objets, ce qui donne des modèles dégénérés. Ce n'est pas une proportion supposée : seulement un minimum pour qu'un « segment » ait un sens. |
| **Arrêt** | dès que la partition ne change plus, ou après `MAX_ITER` itérations | Un seuil fixe de précision était arbitraire, et jamais atteint sur les fichiers bruités. L'arrêt sur un jeu de validation coûtait des données sans gain mesurable : la précision de validation ne passait pas par un maximum. |
| **Polish** | à la fin, chaque modèle est réappris plus longtemps sur sa partition finale ; on garde l'ancien modèle s'il est meilleur | Les apprentissages pendant les itérations sont courts, pour aller vite. Un dernier apprentissage plus long améliore les modèles livrés. |
| **Choix du départ** | celui qui explique le plus d'objets d'apprentissage | C'est la seule mesure disponible sans sacrifier de données. Elle est optimiste, mais de la même façon pour tous les départs. |
| **Objets ambigus** (option `share_ambiguous`) | `False` : chaque objet appartient à un seul segment. `True` : un objet bien classé par les deux modèles sert à apprendre les deux | Pour ces objets, $(x, y)$ est compatible avec les deux segments, et les données ne permettent pas de les attribuer. Les partager donne plus de données à chaque modèle sans attribution arbitraire. C'est l'équivalent « dur » de l'affectation douce d'EM. **Évaluée en section 5.1 : on garde `False`.** |

**Abandonné après les essais** : l'exclusion des objets ambigus de l'apprentissage. Ni la précision de test ni les proportions ne s'amélioraient de façon reproductible, et cela ajoutait de la complexité.

In [6]:
def scores(models, X, y):
    E = np.stack([np.abs(predict(m, X) - y) for m in models], axis=1)     # (N, 2) écart de catégorie
    M = np.stack([margins(m, X, y) for m in models], axis=1)              # (N, 2) marge
    return E, M

def reassign(models, X, y, seg):
    E, M = scores(models, X, y)
    key = E - 1e-3 * np.clip(M, -1, 1)               # lexicographique : l'écart domine, la marge départage
    best = key.argmin(axis=1)
    rows = np.arange(len(X))
    move = key[rows, best] < key[rows, seg] - 1e-12  # strictement mieux expliqué ailleurs
    return np.where(move, best, seg), E

def fit_segments(X, y, p, seg, shared, rng, seconds):
    models = []
    for k in (0, 1):
        idx = np.where((seg == k) | shared)[0]
        size_min = int(max(10, MIN_FRAC * len(X)))
        if len(idx) < size_min:                       # segment trop petit : tirage aléatoire
            idx = rng.choice(len(X), size=size_min, replace=False)
        models.append(learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=seconds))
    return models

def init_partition(X, y, p, how, rng, seconds):
    if how == "random":
        return rng.integers(0, 2, len(X))
    single = learn_mrsort(X, y, p, seed=int(rng.integers(1e9)), seconds=seconds)
    return (predict(single, X) != y).astype(int)      # "errors" : erreurs du modèle unique -> segment 1

def learn_2_mrsort(X, y, p, init="errors", seed=0, share_ambiguous=False,
                   max_iter=MAX_ITER, iter_seconds=ITER_SECONDS, polish_seconds=POLISH_SECONDS):
    rng = np.random.default_rng(seed)
    n = X.shape[1]
    t_iter, t_polish = budget(iter_seconds, n, p), budget(polish_seconds, n, p)
    seg = init_partition(X, y, p, init, rng, t_iter)
    shared = np.zeros(len(X), bool)
    best, history = None, []

    for it in range(max_iter):
        models = fit_segments(X, y, p, seg, shared, rng, t_iter)          # apprentissage
        new_seg, E = reassign(models, X, y, seg)                           # réaffectation
        score = (E.min(axis=1) == 0).mean()                                # expliqués par au moins un modèle
        sizes = np.bincount(new_seg, minlength=2)
        history.append(dict(iteration=it + 1, expliques=score, petit_segment=sizes.min() / sizes.sum(),
                            ambigus=(E == 0).all(axis=1).mean(), changements=(new_seg != seg).mean()))
        if best is None or score > best["score"]:
            best = dict(score=score, models=models, seg=new_seg.copy(), E=E)
        if (new_seg == seg).all():                                         # arrêt : partition stable
            break
        seg = new_seg
        if share_ambiguous:
            shared = (E == 0).all(axis=1)

    # polish : apprentissage plus long sur la partition retenue
    seg = best["seg"]
    shared = (best["E"] == 0).all(axis=1) if share_ambiguous else np.zeros(len(X), bool)
    polished = []
    for k in (0, 1):
        idx = np.where((seg == k) | shared)[0]
        old = best["models"][k]
        if len(idx) < 10:
            polished.append(old)
            continue
        new = learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=t_polish)
        better = (predict(new, X[idx]) == y[idx]).mean() >= (predict(old, X[idx]) == y[idx]).mean()
        polished.append(new if better else old)
    new_seg, E = reassign(polished, X, y, seg)
    score = (E.min(axis=1) == 0).mean()
    if score >= best["score"]:
        best.update(score=score, models=polished, seg=new_seg, E=E)

    best["history"] = pd.DataFrame(history)
    best["init"] = init
    return best

def learn_2_mrsort_multistart(X, y, p, inits=INITS, seed=0, **kwargs):
    runs = [learn_2_mrsort(X, y, p, init=how, seed=seed * 100 + r, **kwargs) for r, how in enumerate(inits)]
    return max(runs, key=lambda r: r["score"]), runs

## 4. Exemple sur un fichier

In [7]:
csv = DATA_DIR / "type_a/cr06_ca03/n0600_rep75-25_noise00.csv"
X, y, p = load_dataset(csv)
print(file_info(csv))

t = time.time()
best, runs = learn_2_mrsort_multistart(X, y, p, seed=0)
print(f"2 modèles : {best['score']:.3f} d'objets expliqués (départ retenu : {best['init']}, {time.time() - t:.0f} s)")
for r in runs:
    print(f"   départ {r['init']:6s} -> {r['score']:.3f}  ({len(r['history'])} itérations)")
print("tailles des segments :", np.bincount(best["seg"], minlength=2).tolist(),
      f"| objets expliqués par les deux modèles : {(best['E'] == 0).all(axis=1).mean():.1%}")
best["history"].round(3)

{'type': 'type_a', 'dossier': 'cr06_ca03', 'fichier': 'n0600_rep75-25_noise00', 'N': 600, 'rep': '75-25', 'petit_attendu': 0.25, 'bruit': 0.0}
2 modèles : 0.988 d'objets expliqués (départ retenu : errors, 194 s)
   départ errors -> 0.988  (20 itérations)
   départ random -> 0.987  (20 itérations)
   départ random -> 0.987  (20 itérations)
   départ random -> 0.968  (20 itérations)
tailles des segments : [357, 243] | objets expliqués par les deux modèles : 23.5%


,iteration,expliques,petit_segment,ambigus,changements
0,1,0.953,0.292,0.382,0.132
1,2,0.950,0.382,0.397,0.153
2,3,0.960,0.277,0.418,0.188
3,4,0.938,0.427,0.423,0.190
4,5,0.943,0.433,0.413,0.063
5,6,0.953,0.442,0.423,0.095
6,7,0.965,0.405,0.432,0.110
7,8,0.962,0.415,0.415,0.057
8,9,0.967,0.410,0.403,0.102
9,10,0.963,0.370,0.402,0.107


In [8]:
for k, model in enumerate(best["models"]):
    show_model(model, f"segment {k} ({(best['seg'] == k).sum()} objets)")

--- segment 0 (357 objets)   lambda = 0.444


,c1,c2,c3,c4,c5,c6
b1,0.206,0.202,0.561,0.478,0.446,0.179
b2,0.229,0.489,0.561,0.480,0.608,0.328
poids,0.222,0.111,0.111,0.222,0.000,0.333


--- segment 1 (243 objets)   lambda = 0.667


,c1,c2,c3,c4,c5,c6
b1,0.063,0.070,0.294,0.328,0.001,0.906
b2,0.543,0.264,0.921,0.784,0.431,0.927
poids,0.333,0.333,0.000,0.000,0.333,0.000


## 5. Benchmark

**Découpage.** Pour chaque fichier et chaque graine, on tire au hasard **10 % des objets comme jeu de test**. Ils ne servent à aucune décision : ni apprentissage, ni réaffectation, ni choix du départ.

**Mesures**

| Mesure | Définition | Rôle |
|---|---|---|
| `deux_train` | part des objets d'apprentissage bien classés par au moins un des deux modèles | ce que l'algorithme optimise (optimiste) |
| `deux_test` | même mesure sur le jeu de test | généralisation |
| `ecart` | `deux_train − deux_test` | surapprentissage |
| `unique_test` | précision d'un **seul** modèle MR-Sort appris sur tout l'apprentissage | référence 1 : ce qu'apportent 2 modèles |
| `moities_test` | `deux_test` pour 2 modèles appris sur une partition **aléatoire**, sans réaffectation | référence 2 : ce qu'apporte la segmentation elle-même |
| `petit_segment` | part du plus petit segment, à comparer à `petit_attendu` (lu dans `rep`) | retrouve-t-on les proportions ? |

**À garder en tête**
- `deux_test` est optimiste : pour chaque objet, on prend le modèle qui le classe bien, ce qui suppose de connaître sa catégorie. **La segmentation n'a de valeur que si `deux_test` dépasse nettement `moities_test`.**
- Le jeu de test est petit : 60 objets pour un fichier de 600, soit 1,7 point par objet. On fait donc 3 graines et on donne la moyenne ± l'écart-type.
- Les références utilisent le temps du polish (`POLISH_SECONDS`), pour que la comparaison soit équitable.

In [9]:
def split(N, seed, test_frac=0.10):
    rng = np.random.default_rng(10_000 + seed)
    test = np.zeros(N, bool)
    test[rng.choice(N, round(test_frac * N), replace=False)] = True
    return test

def two_models_accuracy(models, X, y):
    return np.stack([predict(m, X) == y for m in models]).any(axis=0).mean()

def bench_file(csv, seed, variants=(False, True), keep=None):
    X, y, p = load_dataset(csv)
    test = split(len(X), seed)
    Xtr, ytr, Xte, yte = X[~test], y[~test], X[test], y[test]
    rng = np.random.default_rng(seed)
    t_ref = budget(POLISH_SECONDS, X.shape[1], p)

    single = learn_mrsort(Xtr, ytr, p, seed=seed, seconds=t_ref)                       # référence 1
    halves_seg = rng.integers(0, 2, len(Xtr))                                           # référence 2
    halves = [learn_mrsort(Xtr[halves_seg == k], ytr[halves_seg == k], p, seed=seed + k, seconds=t_ref)
              for k in (0, 1)]
    refs = dict(unique_train=(predict(single, Xtr) == ytr).mean(), unique_test=(predict(single, Xte) == yte).mean(),
                moities_test=two_models_accuracy(halves, Xte, yte))

    rows = []
    for share in variants:
        t = time.time()
        best, _ = learn_2_mrsort_multistart(Xtr, ytr, p, seed=seed, share_ambiguous=share)
        sizes = np.bincount(best["seg"], minlength=2)
        rows.append(dict(**file_info(csv), graine=seed, partage_ambigus=share, **refs,
                         deux_train=best["score"], deux_test=two_models_accuracy(best["models"], Xte, yte),
                         petit_segment=sizes.min() / sizes.sum(),
                         ambigus=(best["E"] == 0).all(axis=1).mean(),
                         depart=best["init"], temps=time.time() - t))
        if keep is not None:                                     # pour afficher les modèles ensuite
            keep.append(dict(fichier=csv, graine=seed, partage_ambigus=share, best=best))
    return rows

def run_bench(files, seeds, variants, out_csv=None, keep=None):
    rows = []
    for f in files:
        for s in seeds:
            rows += bench_file(f, s, variants, keep)
            if out_csv:
                pd.DataFrame(rows).to_csv(out_csv, index=False)  # sauvegarde au fil de l'eau
            r = rows[-1]
            print(f"{r['type']}/{r['dossier']}/{r['fichier']} graine={s} : "
                  + " | ".join(f"partage={x['partage_ambigus']} test={x['deux_test']:.3f}" for x in rows[-len(variants):]),
                  flush=True)
    return pd.DataFrame(rows)

SUMMARY_COLS = ["deux_train", "deux_test", "ecart", "unique_test", "moities_test",
                "gain_vs_unique", "gain_vs_moities", "erreur_proportion", "temps"]

def add_measures(df):
    return df.assign(ecart=df.deux_train - df.deux_test,
                     gain_vs_unique=df.deux_test - df.unique_test,
                     gain_vs_moities=df.deux_test - df.moities_test,
                     erreur_proportion=(df.petit_segment - df.petit_attendu).abs())

def summary(df, by, with_means=False):
    agg = add_measures(df).groupby(by)[SUMMARY_COLS].agg(["mean", "std"])
    fmt = lambda m, s: f"{m:.3f}" if np.isnan(s) else f"{m:.3f} ± {s:.3f}"
    table = pd.DataFrame({c: [fmt(m, s) for m, s in zip(agg[c]["mean"], agg[c]["std"])]
                          for c in SUMMARY_COLS}, index=agg.index)
    return (table, agg.xs("mean", axis=1, level=1)) if with_means else table

### Mise en relief des résultats

Les tableaux de résultats sont colorés pour qu'on lise d'un coup d'œil ce qui compte :
- **`deux_test`** (en gras) : du rouge au vert selon la précision ;
- **`gain_vs_unique` / `gain_vs_moities`** : vert si 2 modèles font mieux que la référence, rouge sinon ;
- **`erreur_proportion`** : vert si on retrouve la bonne taille de segment.

Les couleurs sont calculées sur les moyennes ; le texte affiche la moyenne ± l'écart-type. Si `jinja2` n'est pas installé, les tableaux s'affichent sans couleurs.

In [ ]:
try:
    import jinja2                                     # requis par pandas.Styler
    HAS_STYLE = True
except ImportError:
    HAS_STYLE = False

def show(table, styler=None):
    display(styler if (HAS_STYLE and styler is not None) else table)

def _symmetric(values):
    v = np.nanmax(np.abs(np.asarray(values, float)))
    return dict(vmin=-v, vmax=v) if v > 0 else dict(vmin=-1, vmax=1)

def style_summary(df, by):
    """Tableau 'moyenne ± écart-type', coloré d'après les moyennes."""
    table, means = summary(df, by, with_means=True)
    if not HAS_STYLE:
        return table, None
    st = table.style.background_gradient(cmap="RdYlGn", subset=["deux_test"], gmap=means[["deux_test"]],
                                         axis=None, vmin=0.5, vmax=1.0)
    for c in ("gain_vs_unique", "gain_vs_moities"):
        st = st.background_gradient(cmap="RdYlGn", subset=[c], gmap=means[[c]], axis=None, **_symmetric(means[c]))
    st = st.background_gradient(cmap="RdYlGn_r", subset=["erreur_proportion"], gmap=means[["erreur_proportion"]],
                                axis=None, vmin=0, vmax=0.25)
    return table, st.set_properties(subset=["deux_test"], **{"font-weight": "bold"})

DETAIL_COLS = ["type", "dossier", "fichier", "graine", "unique_test", "moities_test", "deux_train", "deux_test",
               "petit_attendu", "petit_segment", "ambigus", "depart", "temps"]

def style_detail(df):
    """Une ligne par (fichier, graine)."""
    d = add_measures(df)
    table = d[DETAIL_COLS + ["gain_vs_unique", "gain_vs_moities"]].reset_index(drop=True)
    if not HAS_STYLE:
        return table, None
    num = table.select_dtypes("number").columns.drop(["graine"])
    st = (table.style.format({c: "{:.3f}" for c in num}).format({"temps": "{:.0f} s"})
          .background_gradient(cmap="RdYlGn", subset=["deux_test"], vmin=0.5, vmax=1.0)
          .set_properties(subset=["deux_test"], **{"font-weight": "bold"}))
    for c in ("gain_vs_unique", "gain_vs_moities"):
        st = st.background_gradient(cmap="RdYlGn", subset=[c], **_symmetric(table[c]))
    return table, st

### 5.1 Phase 1 : partager ou non les objets ambigus

On compare les deux valeurs de `share_ambiguous`. Le plan prévoyait **12 fichiers de 600 objets** (2 par dossier, chaque combinaison répartition × bruit représentée 3 fois) et 3 graines par fichier. Les résultats enregistrés dans `bench_phase1.csv` portent sur **le premier fichier et 2 graines**.

Par défaut la cellule relit `bench_phase1.csv` ; passer `RUN_PHASE1 = True` pour relancer les essais.

In [ ]:
folders = sorted(DATA_DIR.glob("type_*/cr*_ca*"))
PHASE1_FILES = []
for i, folder in enumerate(folders):
    names = (["n0600_rep75-25_noise00", "n0600_rep50-50_noise10"] if i % 2 == 0
             else ["n0600_rep50-50_noise00", "n0600_rep75-25_noise10"])
    PHASE1_FILES += [folder / f"{name}.csv" for name in names]

RUN_PHASE1 = False
phase1 = (run_bench(PHASE1_FILES, seeds=[0, 1, 2], variants=(False, True), out_csv="bench_phase1.csv")
          if RUN_PHASE1 else pd.read_csv("bench_phase1.csv"))
print(f"{phase1.fichier.nunique()} fichier(s), graines {sorted(phase1.graine.unique().tolist())}, {len(phase1)} essais")

# 1) essai par essai : sans / avec partage côte à côte
LABELS = {False: "sans partage", True: "avec partage"}
d = add_measures(phase1).assign(partage=lambda t: t.partage_ambigus.map(LABELS))
per_run = d.pivot_table(index=["fichier", "graine"], columns="partage",
                        values=["deux_test", "petit_segment", "temps"])
per_run = per_run.reindex(columns=list(LABELS.values()), level=1)

# 2) moyennes : la meilleure option de chaque ligne est mise en relief
HIGHER_IS_BETTER = {"deux_train": True, "deux_test": True, "ecart": False, "erreur_proportion": False,
                    "ambigus": None, "temps": False}
comp = d.groupby("partage")[list(HIGHER_IS_BETTER)].mean().T[list(LABELS.values())]
comp["écart (avec − sans)"] = comp["avec partage"] - comp["sans partage"]

def highlight_best(row):
    sens = HIGHER_IS_BETTER[row.name]
    out = ["" for _ in row]
    if sens is not None:
        vals = row[list(LABELS.values())]
        win = vals.idxmax() if sens else vals.idxmin()
        out[list(row.index).index(win)] = "background-color: #c6efce; font-weight: bold"
    return out

if HAS_STYLE:
    show(per_run, per_run.style.format("{:.3f}").format("{:.0f} s", subset=["temps"])
         .highlight_max(subset=["deux_test"], axis=1, props="background-color: #c6efce; font-weight: bold"))
    show(comp, comp.style.format("{:.3f}").format("{:+.3f}", subset=["écart (avec − sans)"])
         .format("{:.0f} s", subset=pd.IndexSlice[["temps"], list(LABELS.values())])
         .apply(highlight_best, axis=1))
else:
    show(per_run.round(3)); show(comp.round(3))

SHARE_AMBIGUOUS = False          # option retenue (voir conclusion ci-dessous)

**Conclusion de la phase 1 : on garde `share_ambiguous = False`.**

Sur les essais enregistrés, partager les objets ambigus ne fait pas mieux :
- **précision de test** : 0,958 sans partage contre 0,933 avec, en moyenne ; chaque option gagne un des deux essais (0,950 / 0,883 pour la graine 0, 0,967 / 0,983 pour la graine 1). L'écart n'est donc pas significatif ;
- **proportions** : l'erreur sur la taille du petit segment est comparable (0,10 sans, 0,11 avec) ;
- **temps** : le partage coûte environ 23 % de temps en plus, car chaque modèle apprend sur plus d'objets.

À performances égales, on retient la version la plus simple et la plus rapide : chaque objet appartient à un seul segment.

### 5.2 Phase 2 : tous les fichiers, sans partage des ambigus

Le benchmark complet porte sur les **72 fichiers** avec `SHARE_AMBIGUOUS = False` et une graine. Il est long : par défaut la cellule relit `bench_complet.csv` (sauvegardé au fil de l'eau) et indique combien de fichiers sont déjà couverts. Passer `RUN_FULL = True` pour le relancer.

In [ ]:
RUN_FULL = False

ALL_FILES = sorted(DATA_DIR.glob("type_*/cr*_ca*/*.csv"))
full = (run_bench(ALL_FILES, seeds=[2], variants=(SHARE_AMBIGUOUS,), out_csv="bench_complet.csv")
        if RUN_FULL else pd.read_csv("bench_complet.csv"))
full = full[full.partage_ambigus == SHARE_AMBIGUOUS]
print(f"{full[['type', 'dossier', 'fichier']].drop_duplicates().shape[0]} fichiers sur {len(ALL_FILES)} couverts")

print("\n— Vue d'ensemble")
show(*style_summary(full.assign(ensemble="tous les fichiers"), "ensemble"))
print("— Par dossier (nombre de critères / catégories)")
show(*style_summary(full, ["type", "dossier"]))
print("— Par répartition et bruit")
show(*style_summary(full, ["rep", "bruit"]))
print("— Par taille du learning set")
show(*style_summary(full, "N"))

## 6. Lancer l'algorithme sur un fichier ou sur un dossier

Mettre dans `CHEMIN` :
- **un fichier `.csv`** : on affiche ses mesures et les deux modèles appris ;
- **un dossier** : tous les `.csv` qu'il contient, sous-dossiers compris, sont traités (par exemple `DATA_DIR / "type_b"` ou `DATA_DIR` pour tout le jeu). On affiche une ligne par fichier, puis un résumé par dossier.

Chaque `.csv` doit avoir son `problem.yml` dans le même dossier. Le protocole est celui de la section 5 : 10 % des objets en test, `share_ambiguous = False`, références « modèle unique » et « moitiés aléatoires ». Les résultats sont aussi écrits dans `SORTIE` (mettre `None` pour ne rien écrire).

In [ ]:
CHEMIN  = DATA_DIR / "type_b/cr06_ca03/n0600_rep75-25_noise00.csv"    # un fichier...
# CHEMIN = DATA_DIR / "type_b/cr06_ca03"                               # ... ou un dossier
GRAINES = [0]
SORTIE  = "resultats_run.csv"

CHEMIN = Path(CHEMIN)
files = [CHEMIN] if CHEMIN.is_file() else sorted(CHEMIN.rglob("*.csv"))
assert files, f"aucun fichier .csv trouvé dans {CHEMIN}"
print(f"{len(files)} fichier(s) × {len(GRAINES)} graine(s), share_ambiguous = {SHARE_AMBIGUOUS}\n")

kept = []
res = run_bench(files, GRAINES, variants=(SHARE_AMBIGUOUS,), out_csv=SORTIE, keep=kept)

print("\n— Résultats par fichier")
show(*style_detail(res))
if len(files) > 1:
    print("— Résumé par dossier")
    show(*style_summary(res, ["type", "dossier"]))
else:
    for run in kept:
        best = run["best"]
        print(f"\n— Modèles appris (graine {run['graine']}, départ retenu : {best['init']})")
        for k, model in enumerate(best["models"]):
            show_model(model, f"segment {k} ({(best['seg'] == k).sum()} objets d'apprentissage)")

### Lexique des colonnes

| Colonne | Signification |
|---|---|
| `type`, `dossier`, `fichier` | jeu de données (`cr06_ca03` = 6 critères, 3 catégories ; `n0600_rep75-25_noise10` = 600 objets, segments 75/25, 10 % de bruit) |
| `graine` | graine aléatoire (tirage du test + apprentissages) |
| `unique_test` | précision en test d'un **seul** modèle MR-Sort (référence 1) |
| `moities_test` | précision en test de 2 modèles appris sur une partition **aléatoire** (référence 2) |
| `deux_train` / `deux_test` | part des objets bien classés par au moins un des 2 modèles, en apprentissage / en test |
| `petit_attendu` | taille attendue du petit segment, lue dans `rep` (évaluation seulement) |
| `petit_segment` | taille du petit segment trouvé (part de l'apprentissage) |
| `ambigus` | part des objets d'apprentissage bien classés par les **deux** modèles |
| `depart` | départ retenu : `errors` (guidé) ou `random` |
| `temps` | durée de l'algorithme en secondes (hors références) |
| `ecart` | `deux_train − deux_test` (surapprentissage) |
| `gain_vs_unique` | `deux_test − unique_test` : apport de 2 modèles |
| `gain_vs_moities` | `deux_test − moities_test` : apport de la segmentation — **la mesure clé** |
| `erreur_proportion` | `|petit_segment − petit_attendu|` |

Précisions entre 0 et 1. `deux_test` est optimiste (on prend le modèle qui classe bien l'objet) : seul un `gain_vs_moities` nettement positif montre une vraie segmentation. Avec 60 objets de test, 1 objet ≈ 1,7 point.